# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
Rule Definition: A page is flagged for an urgent content refresh if it is stale (days since last update >= 180) and has meaningful search visibility (impressions >= 500). The score is calculated as stale * visible * impressions_90d.
Reason Codes:
- REFRESH_STALE_HIGH_IMPRESSION: Assigned when a stale page receives significant search traffic and needs an update.
- NO_ACTION: Assigned when the page does not meet the staleness or visibility threshold.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np
import urllib.request

file_path = "content_refresh_anonymized.csv"
data_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

if not os.path.exists(file_path):
    urllib.request.urlretrieve(data_url, file_path)

df = pd.read_csv(file_path)
print(f"Dataset loaded successfully with {len(df):,} rows.")

Dataset loaded successfully with 30,000 rows.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
Building the Queue: We compute the baseline score using our hand rule logic, assign reason codes and action labels, sort by descending score priority, and write the resulting queue to work/outputs/baseline_action_score.csv.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Encode hand rule score, reason codes, and action labels
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

df["baseline_score"] = stale * visible * df["impressions_90d"]
df["reason_code"] = np.where(df["baseline_score"] > 0, "REFRESH_STALE_HIGH_IMPRESSION", "NO_ACTION")
df["action_label"] = np.where(df["baseline_score"] > 0, "Urgent Content Refresh", "Monitor")

# Sort queue by score descending
ranked_queue = df.sort_values("baseline_score", ascending=False)

# Write queue to CSV in work/outputs/
os.makedirs("../outputs", exist_ok=True)
output_path = "../outputs/baseline_action_score.csv"
ranked_queue.to_csv(output_path, index=False)
print(f"Baseline action score queue successfully written to: {output_path}")

Baseline action score queue successfully written to: ../outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
Top-20 Detailed Review:
1-20: Action: Urgent Content Refresh | Reason Code: REFRESH_STALE_HIGH_IMPRESSION | Confidence: High impression volume with 180+ days of staleness. | Wrong if: Seasonal demand surge or evergreen topic that requires no textual change.
(Note: All top 20 items share this high-exposure profile by rule design; individual inspection ensures no structural data corruption).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display top 20 rows from the ranked queue for visual review
display_cols = ["baseline_score", "reason_code", "action_label", "impressions_90d", "days_since_last_update", "avg_position", "ctr"]
top20 = ranked_queue[display_cols].head(20)
print("=== Top 20 Ranked Baseline Queue Preview ===")
display(top20)

=== Top 20 Ranked Baseline Queue Preview ===


,baseline_score,reason_code,action_label,impressions_90d,days_since_last_update,avg_position,ctr
16751,61678,REFRESH_STALE_HIGH_IMPRESSION,Urgent Content Refresh,61678,194,19.7,0.15
16514,59472,REFRESH_STALE_HIGH_IMPRESSION,Urgent Content Refresh,59472,194,24.8,0.13
7021,25715,REFRESH_STALE_HIGH_IMPRESSION,Urgent Content Refresh,25715,194,22.2,0.23
21268,13299,REFRESH_STALE_HIGH_IMPRESSION,Urgent Content Refresh,13299,193,10.5,0.49
11489,7812,REFRESH_STALE_HIGH_IMPRESSION,Urgent Content Refresh,7812,194,39.0,0.01
12045,7558,REFRESH_STALE_HIGH_IMPRESSION,Urgent Content Refresh,7558,193,17.9,0.20
698,4590,REFRESH_STALE_HIGH_IMPRESSION,Urgent Content Refresh,4590,194,31.0,0.00
5327,4556,REFRESH_STALE_HIGH_IMPRESSION,Urgent Content Refresh,4556,194,16.4,0.33
26810,4429,REFRESH_STALE_HIGH_IMPRESSION,Urgent Content Refresh,4429,194,25.3,0.38
20837,1697,REFRESH_STALE_HIGH_IMPRESSION,Urgent Content Refresh,1697,193,15.8,0.12


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
Weak Picks & Leakage Audit:
- Weak Picks: The hand rule relies heavily on raw impression volume, meaning pages with lower traffic but severe percentage decay are pushed down the queue.
- Leakage Check: Confirmed zero data leakage. Features used (days_since_last_update, impressions_90d) are strictly observable past metrics; no target labels or future trend percentages were included in the rule.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify no target or leakage columns exist in our scoring feature subset
forbidden_cols = ["trend_pct", "is_declining_label"]
leak_check = any(col in ["days_since_last_update", "impressions_90d"] for col in forbidden_cols)
print(f"Leakage check passed (No forbidden columns used): {not leak_check}")

Leakage check passed (No forbidden columns used): True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.